# mlp-study-kit — Introduction to `nn_core`

This notebook walks through the full pipeline:
1. Install & verify the package
2. Build and inspect a network
3. Generate training data
4. Train with early stopping
5. Visualise predictions
6. Save and reload weights

**No path hacks needed** after `pip install -e .`

## 0. Setup

In [ ]:
# Run once if needed
# !pip install -e .. --quiet

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')          # safe for notebooks without display
import matplotlib.pyplot as plt

from nn_core import NeuralNetwork, ActivationFn, LossFn, ObjLogger

logger = ObjLogger("Notebook")
logger("nn_core imported successfully", color="green")

## 1. Build a Network

In [ ]:
np.random.seed(42)

structure = [
    {"type": "input",  "units": 1},
    {"type": "dense",  "units": 32, "activation_function": "tanh",   "bias": True},
    {"type": "dense",  "units": 16, "activation_function": "tanh",   "bias": True},
    {"type": "dense",  "units": 1,  "activation_function": "linear", "bias": True},
]

model = NeuralNetwork()
net   = model.create_network(structure)

print(model)   # human-readable architecture

## 2. Generate Training Data

Target function: $y = \sin(2x) + \cos(x) + 5 + \text{noise}$

In [ ]:
rng = np.random.default_rng(42)
n   = 120

X_all = np.linspace(-3, 3, n).reshape(-1, 1)
Y_all = np.sin(2 * X_all) + np.cos(X_all) + 5.0 + rng.normal(0, 0.15, (n, 1))

# 80/20 train-test split
split = int(0.8 * n)
idx   = rng.permutation(n)
X_train, Y_train = X_all[idx[:split]], Y_all[idx[:split]]
X_test,  Y_test  = X_all[idx[split:]], Y_all[idx[split:]]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(X_train, Y_train, s=15, label="Train", alpha=0.7)
ax.scatter(X_test,  Y_test,  s=15, label="Test",  alpha=0.7, color="orange")
ax.set_title("Training data")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig("data_preview.png", dpi=80)
plt.show()
print("Plot saved to data_preview.png")

## 3. Train with Early Stopping

In [ ]:
final_loss = model.train(
    net, X_train, Y_train,
    x_test=X_test, y_test=Y_test,
    l_rate=0.02,
    n_epoch=1000,
    loss_function="mse",
    epsilon=1e-5,         # stop when improvement < this
    verbose=1,
    save_plot="loss_history.png",
)

print(f"\nFinal train loss: {final_loss:.6f}")
print("Loss plot saved to loss_history.png")

## 4. Visualise Predictions

In [ ]:
X_dense = np.linspace(-3.5, 3.5, 300).reshape(-1, 1)
preds   = np.array(model.predict(net, X_dense))

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(X_train, Y_train, s=12, label="Train", alpha=0.6)
ax.scatter(X_test,  Y_test,  s=12, label="Test",  alpha=0.6, color="orange")
ax.plot(X_dense, preds, color="red", lw=2, label="Prediction")
ax.set_title("Predictions after training")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig("predictions.png", dpi=80)
plt.show()
print("Plot saved to predictions.png")

## 5. Save and Reload Weights

In [ ]:
# Save
model.save_weights(net, "trained_weights.npy")

# Build a fresh network with the same structure
net2 = model.create_network(structure)
preds_before = np.array(model.predict(net2, X_test[:3]))

# Load weights
model.load_weights(net2, "trained_weights.npy")
preds_after = np.array(model.predict(net2, X_test[:3]))

print("Predictions before loading:", preds_before.flatten().round(4))
print("Predictions after  loading:", preds_after.flatten().round(4))
print("Match:", np.allclose(preds_after, model.predict(net, X_test[:3])))

## 6. Activations Explorer

In [ ]:
af = ActivationFn()
v  = np.linspace(-4, 4, 200)

activations = ["sigmoid", "tanh", "relu", "leaky_relu", "elu"]

fig, axes = plt.subplots(1, len(activations), figsize=(16, 3), sharey=False)
for ax, name in zip(axes, activations):
    layer = {"activation_potential": v, "output": None}
    y = af.output(layer, name)
    layer["output"] = y
    dy = af.output(layer, name, derivative=True)
    ax.plot(v, y,  label="f(v)",  lw=2)
    ax.plot(v, dy, label="f'(v)", lw=2, linestyle="--")
    ax.set_title(name)
    ax.legend(fontsize=8)
    ax.grid(True)
    ax.axhline(0, color="k", lw=0.5)
    ax.axvline(0, color="k", lw=0.5)

plt.suptitle("Activation Functions (forward + derivative)", fontsize=13)
plt.tight_layout()
plt.savefig("activations.png", dpi=80)
plt.show()
print("Plot saved to activations.png")

## Next Steps

- Run `exercises/ex09_full_backprop.py` to see the algorithm built from scratch
- Run `tools/backprop_debugger.py` to step through one iteration in lecture notation
- Explore `experiments/20251224_v2.py` for Glorot init + Adagrad optimizer
- Read `tools/README.md` to choose the right debug tool for exam prep